In [1]:
# Import libraries
import pandas as pd
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# Function to load datasets
def load_dataset(name):
    if name.lower() == "iris":
        data = load_iris()
    elif name.lower() == "wine":
        data = load_wine()
    else:
        raise ValueError("Dataset not supported.")
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = pd.Series(data.target)
    return X, y

In [3]:
# Function to evaluate model
def evaluate_model(clf, X_train, X_test, y_train, y_test):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted')
    }

In [4]:
# Function to perform cross-validation
def cv_scores(clf, X, y, cv):
    scoring = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']
    scores = {}
    for metric in scoring:
        scores[metric] = cross_val_score(clf, X, y, cv=cv, scoring=metric).mean()
    return scores

In [5]:
# Classifiers
classifiers = {
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

# Datasets
datasets = ["iris", "wine"]

In [10]:
# Run evaluation
for dataset_name in datasets:
    print(f"\n=== Dataset: {dataset_name.upper()} ===")
    X, y = load_dataset(dataset_name)

    # Preprocess
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    # Holdout evaluations
    for test_size, label in [(0.2, "80-20 split"), (1/3, "2/3 - 1/3 split")]:
        X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=test_size, random_state=42, stratify=y)
        print(f"\n-- Holdout ({label}) --")
        for name, clf in classifiers.items():
            scores = evaluate_model(clf, X_train, X_test, y_train, y_test)
            print(f"{name}: {scores}")
    # Cross-validation evaluations
    for folds in [10, 5]:
        print(f"\n-- {folds}-Fold Cross-Validation --")
        for name, clf in classifiers.items():
            scores = cv_scores(clf, X_scaled, y, cv=folds)
            print(f"{name}: {scores}")



=== Dataset: IRIS ===

-- Holdout (80-20 split) --
Naive Bayes: {'Accuracy': 0.9666666666666667, 'Precision': 0.9696969696969696, 'Recall': 0.9666666666666667, 'F1': 0.9665831244778613}
KNN: {'Accuracy': 0.9333333333333333, 'Precision': 0.9444444444444445, 'Recall': 0.9333333333333333, 'F1': 0.9326599326599326}
Decision Tree: {'Accuracy': 0.9333333333333333, 'Precision': 0.9333333333333333, 'Recall': 0.9333333333333333, 'F1': 0.9333333333333333}

-- Holdout (2/3 - 1/3 split) --
Naive Bayes: {'Accuracy': 0.92, 'Precision': 0.9236491228070176, 'Recall': 0.92, 'F1': 0.9197222222222223}
KNN: {'Accuracy': 0.92, 'Precision': 0.9352380952380952, 'Recall': 0.92, 'F1': 0.9188771929824562}
Decision Tree: {'Accuracy': 0.94, 'Precision': 0.9490000000000001, 'Recall': 0.94, 'F1': 0.9395292066259807}

-- 10-Fold Cross-Validation --
Naive Bayes: {'accuracy': np.float64(0.9533333333333334), 'precision_weighted': np.float64(0.9626984126984126), 'recall_weighted': np.float64(0.9533333333333334), 'f1_we